# Pipeline Definitivo — IS_FRAUD

Pipeline completo: KNN imputation → Feature Engineering → Escalado → Modelos → Persistencia.
**Diseñado para aplicarse directamente a cualquier dataset nuevo.**

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score,
    precision_score, recall_score, make_scorer, fbeta_score)
import xgboost as xgb
import lightgbm as lgb
try:
    import catboost as cb
    CATBOOST_AVAIL = True
except ImportError:
    CATBOOST_AVAIL = False
    print('CatBoost no instalado — se omite')
sys.path.append(str(Path.cwd().parent))
from model.feature_engineering import *
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('OK')

## 1. Pipeline completo como función

Incluye KNNImputer para missing values (3 vecinos), feature engineering con anti-leakage, y entrenamiento del mejor modelo con GridSearchCV.

In [ ]:
def train_pipeline(df, target='IS_FRAUD', test_size=0.2, random_state=42, models=None):
    # Drop del otro target para evitar data leakage
    other = [t for t in ['IS_FRAUD', 'IMPACTO_FRAUDE'] if t != target]
    df = df.drop(columns=other, errors='ignore')
    """
    Pipeline completo de entrenamiento.
    
    Parámetros:
    - df: DataFrame con datos
    - target: nombre de columna target
    - test_size: proporción de test
    - models: lista de (nombre, estimador, param_grid). None = usa default
    
    Retorna:
    - best_model: mejor estimador entrenado
    - fe: FeatureEngineer fit
    - scaler: StandardScaler fit
    - imputer: KNNImputer fit
    - results_df: DataFrame con resultados
    - Xtr, Xte, y_train, y_test: datos transformados
    """
    # 1. Train/Test split
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state,
        stratify=df[target] if df[target].nunique() <= 10 else None
    )
    print(f'Train: {train_df.shape[0]}  Test: {test_df.shape[0]}')
    
    # 2. KNN Imputation (separada para no contaminar FE)
    num_cols_original = train_df.select_dtypes(include=[np.number]).columns.tolist()
    imputer = KNNImputer(n_neighbors=3)
    train_df[num_cols_original] = imputer.fit_transform(train_df[num_cols_original])
    test_df[num_cols_original] = imputer.transform(test_df[num_cols_original])
    print(f'Missing imputados con KNN (k=3)')
    
    # 3. Feature Engineering
    fe = FeatureEngineer(encode_target=target, random_state=random_state)
    X_train = fe.fit_transform(train_df)
    y_train = X_train.pop(target).values
    X_test = fe.transform(test_df)
    y_test = X_test.pop(target).values
    num_feats = X_train.select_dtypes(include=[np.number]).columns.tolist()
    print(f'Features: {X_train.shape[1]}')
    
    # 4. Escalado
    scaler = StandardScaler()
    Xtr = X_train.copy(); Xte = X_test.copy()
    Xtr[num_feats] = scaler.fit_transform(X_train[num_feats])
    Xte[num_feats] = scaler.transform(X_test[num_feats])
    print('Escalado completado')
    
    # 5. Modelos default
    if models is None:
        models = [
            ('LogisticRegression', LogisticRegression(), {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced'], 'max_iter': [1000]}),
            ('RandomForest', RandomForestClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [5, 10, None], 'class_weight': [None, 'balanced']}),
            ('XGBoost', xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss'), {'n_estimators': [100, 200], 'max_depth': [3, 5, 7], 'learning_rate': [0.05, 0.1], 'scale_pos_weight': [1, 3, 5]}),
            ('LightGBM', lgb.LGBMClassifier(random_state=42, verbose=-1), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63], 'class_weight': [None, 'balanced']}),
        ]
        if CATBOOST_AVAIL:
            models.append(('CatBoost', cb.CatBoostClassifier(random_state=42, verbose=False, eval_metric='AUC'),
                           {'iterations': [100, 200], 'depth': [4, 6], 'learning_rate': [0.05, 0.1]}))
    
    # 6. GridSearchCV
    f2 = make_scorer(fbeta_score, beta=2, pos_label=1)
    results = []; bests = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for name, est, params in models:
        print(f'\n>>> {name}')
        gs = GridSearchCV(est, params, cv=cv, scoring=f2, n_jobs=-1, verbose=0)
        gs.fit(Xtr, y_train)
        bests[name] = gs.best_estimator_
        yp = gs.predict(Xte)
        yprob = gs.predict_proba(Xte)[:, 1] if hasattr(gs, 'predict_proba') else None
        r = {'modelo': name, 'cv_auprc': gs.best_score_}
        if yprob is not None:
            r['test_auc'] = roc_auc_score(y_test, yprob)
            r['test_auprc'] = average_precision_score(y_test, yprob)
        results.append(r)
        print(f'  CV AUPRC={gs.best_score_:.4f}', f' AUC={r.get("test_auc",0):.4f}' if 'test_auc' in r else '')
    
    results_df = pd.DataFrame(results).sort_values('test_auprc', ascending=False)
    best_model = bests[results_df.iloc[0]['modelo']]
    print(f'\n--- Mejor: {results_df.iloc[0]["modelo"]} ---')
    
    return best_model, fe, scaler, imputer, results_df, Xtr, Xte, y_train, y_test

print('Función train_pipeline() definida')
print('Ejemplo de uso: best_model, fe, scaler, imputer, res, Xtr, Xte, ytr, yte = train_pipeline(df)')

## 2. Ejecutar pipeline

In [ ]:
DATA_PATH = Path.cwd().parent / 'Notebooks' / 'data' / 'dataset_fraude.csv'
df = pd.read_csv(DATA_PATH)
best_model, fe, scaler, imputer, res, Xtr, Xte, ytr, yte = train_pipeline(df, target='IS_FRAUD')
print('\n' + '='*50)
print('RESULTADOS:')
print(res[['modelo','cv_auprc','test_auc','test_auprc']].round(4).to_string(index=False))

## 3. Evaluación detallada

In [ ]:
best_name = res.iloc[0]['modelo']
best = best_model
print(f'Mejor modelo (mayor F2): {best_name}')

# 1. Out-of-fold raw probabilities for robust threshold
cv5 = StratifiedKFold(5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(best, Xtr, ytr, cv=cv5, method='predict_proba', n_jobs=-1)[:, 1]

# 2. Threshold on raw OOF probs (recall-first: max precision subject to recall >= 90%)
target_recall = 0.90
thrs = np.linspace(0.01, 0.9, 400)
best_t = None; best_p = -1; best_r = 0
for t in thrs:
    yt = (oof_probs >= t).astype(int)
    r = recall_score(ytr, yt)
    p = precision_score(ytr, yt)
    if r >= target_recall and p > best_p:
        best_t, best_p, best_r = t, p, r
print(f'Threshold OOF raw: {best_t:.4f}  Recall={best_r:.4f}  Precision={best_p:.4f}')

# 3. Apply same threshold to test (raw probabilities from best.model)
yprob = best.predict_proba(Xte)[:, 1]
yp_opt = (yprob >= best_t).astype(int)

# 4. Final metrics
print()
print(classification_report(yte, yp_opt, digits=4))
alerts = yp_opt.sum()
alerts_per_100k = alerts / len(Xte) * 100000
print(f'Alertas totales: {alerts}  Alertas/100k: {alerts_per_100k:.1f}')


## 4. Guardar modelo para producción

In [ ]:
import joblib
MODEL_DIR = Path.cwd().parent / 'model' / 'saved_models'
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(best_model, MODEL_DIR / 'best_model_fraud.pkl')
joblib.dump(fe, MODEL_DIR / 'feature_engineer_fraud.pkl')
joblib.dump(scaler, MODEL_DIR / 'scaler_fraud.pkl')
joblib.dump(imputer, MODEL_DIR / 'imputer_fraud.pkl')
print(f'Modelo guardado en {MODEL_DIR}')

## 5. Función de predicción para datos nuevos

In [ ]:
def predict_new(df_new, best_model, fe, scaler, imputer, target='IS_FRAUD'):
    """
    Aplica el pipeline completo a datos nuevos.
    - imputer: KNNImputer fit
    - fe: FeatureEngineer fit
    - scaler: StandardScaler fit
    - best_model: modelo entrenado
    """
    df = df_new.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df[num_cols] = imputer.transform(df[num_cols])
    X = fe.transform(df)
    if target in X.columns:
        X = X.drop(columns=[target])
    num_feats = X.select_dtypes(include=[np.number]).columns.tolist()
    X[num_feats] = scaler.transform(X[num_feats])
    preds = best_model.predict(X)
    probs = best_model.predict_proba(X)[:, 1]
    return preds, probs

df_demo = df.drop(columns=['IMPACTO_FRAUDE'], errors='ignore')
preds, probs = predict_new(df_demo.iloc[:5], best_model, fe, scaler, imputer)
print('Predicciones demo:', preds)
print('Probabilidades:', probs.round(4))
print('Pipeline listo para producción.')